# Rung 06c — the epoch-3 control rung 06 never ran

`context/decisions/epoch-matched-control.md` records the gap this notebook closes: rungs 14
and 15 each evaluated **all three** epochs, but rung 06 stopped at epoch 2 while its
`eval_loss` was still falling (0.3204 → 0.2925 → **0.2782**). Both team rungs therefore
compare their ep3 against rung 06's **ep2** — and rung 15's only apparent win lives exactly
there.

| ckpt | rung 06 | 14 appearance-aug | 15 count-target |
|---|---|---|---|
| 860 (ep1) | 0.5345 | 0.5291 | 0.5274 |
| 1720 (ep2) | **0.5667** | 0.5642 | 0.5612 |
| 2580 (ep3) | **this notebook** | 0.5611 | 0.5699 |

**Pre-registered read, copied from the decision note before any number exists:**
if this lands **≥ 0.5699**, rung 15 dies outright (and we may have gained a free
checkpoint); if it lands **< 0.5667**, the ep3 rows of rungs 14 and 15 both need rereading.

🔴 **Same card on purpose.** ~0.5 % of stored answers change on a GPU swap
(`archived-results-not-bit-reproducible`), so this runs on the **RTX 5090** that produced
rungs 14 and 15. The comparison this notebook exists to serve is therefore within-machine;
the older 0.5667 was measured elsewhere and carries that caveat.

⚠️ **This is not a seed repeat and does not become one.** No run has ever been repeated with
a different seed, so a ±0.003 difference stays formally indistinguishable from noise.

In [ ]:
# papermill parameters
SMOKE = True           # True -> 40 questions, just to prove the wiring. Full run: -p SMOKE False
RUN_TAG = "ep3_full"

In [ ]:
import json, logging, os, shutil, sys, time
from pathlib import Path
import pandas as pd

# `merge_checkpoint` shells out to the bare `swift` binary. A papermill kernel does NOT
# inherit the env's bin/ on PATH, so it must be put there explicitly -- and it must be
# THIS interpreter's bin, so the CLI and the kernel come from the same environment.
_envbin = str(Path(sys.executable).parent)
if _envbin not in os.environ.get("PATH", "").split(os.pathsep):
    os.environ["PATH"] = _envbin + os.pathsep + os.environ.get("PATH", "")
print("swift on PATH:", (Path(_envbin) / "swift").exists(), "->", _envbin)

EXP = Path.cwd()
REPO = EXP
while REPO != REPO.parent and not (REPO / "src").is_dir():
    REPO = REPO.parent
for p in (REPO / "src", EXP / "_models", REPO / "experiments/02-lora-sft/_models"):
    if p.is_dir():
        sys.path.insert(0, str(p))

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s %(message)s",
                    datefmt="%H:%M:%S")

from frame.config import BaselineConfig
from frame.run import run_baseline
from frame import ledger, metrics
from lora_sft_train import LoRAConfig, merge_checkpoint

# ONE arm. 06b was symmetric because its question was "is epoch 1 better in general"; this
# one asks what rung 06 scores at epoch 3, and rung 02's ep3 settles nothing about 14/15.
RUN_DIR = REPO / "experiments/06-vit-lora/runs/06_vit_lora_v1"
CKPT = "checkpoint-2580"    # epoch 3 (2580 steps / 3 epochs)

# The QA parquets live in different places on the pod and on a laptop. Resolve by
# LOOKING for them, and fail loudly if neither has them -- an eval on an empty data root
# is the classic silent zero.
DATA_ROOT = next(
    (d for d in (REPO / "external_data" / "orena-data", Path("/workspace/orena-data"))
     if (d / "heico" / "data" / "frame" / "test.parquet").exists()),
    None,
)
assert DATA_ROOT is not None, "no frame/test.parquet under repo/external_data/orena-data nor /workspace/orena-data"
print("data_root:", DATA_ROOT)
print("repo:", REPO)
print("run_dir:", RUN_DIR, RUN_DIR.exists())

## Gate 0 — the ep3 adapter must exist, and the merge is produced here

Rung 06's `merged/` holds **only** `checkpoint-1720`: rungs 14 and 15 delete each 17 GB merge
once its eval is scored, and rung 06's ep1 merge went the same way. So ep3 is merged here.

There is no rung-06-specific exporter — `_models/vit_lora_train.py` has none, and rung 06's
own `merged/checkpoint-1720` was produced by the shared `lora_sft_train.merge_checkpoint`
(`swift export --merge_lora`, `02-lora-sft/_models/lora_sft_train.py:254`). Using it is
faithful to how the reference number was made, not a substitution.

In [ ]:
merged = RUN_DIR / "merged" / CKPT
if merged.is_dir() and any(merged.iterdir()):
    print(f"OK    merged already present -> {merged}")
    MERGED_HERE = False
else:
    ck = sorted((RUN_DIR / "ckpt").glob(f"v0-*/{CKPT}"))
    assert ck, f"no adapter at {RUN_DIR}/ckpt/v0-*/{CKPT}"
    print(f"MERGE {ck[0]}")
    cfg = LoRAConfig(exp_dir=RUN_DIR.parents[1], run_name=RUN_DIR.name)
    t0 = time.perf_counter()
    merged = merge_checkpoint(cfg, ck[0])
    MERGED_HERE = True
    print(f"      merged in {time.perf_counter()-t0:.0f}s -> {merged}")
merged

## Eval — full 6252, protocol IDENTICAL to `eval_best`

Same judge, same `max_pixels`, same seed, same `run_baseline`, `answer_postprocess` left at
its `None` default. Any deviation would invalidate the comparison against the epoch-2 numbers
already measured **and** against rungs 14/15, which inherited these same constants from rung
06's recipe (`lora_sft_train.py:57,60`).

🔴 **Disk.** The merged 8B bf16 checkpoint is ~17 GB. It is deleted in a `finally` the moment
its `results.csv` is scored — the failure that fills the volume is an eval that RAISED (CUDA
OOM, judge crash) and left 17 GB behind. Rung 12 hit exactly this, mid-run. Only a merge THIS
notebook created is removed; a pre-existing one is left alone.

In [ ]:
t0 = time.perf_counter()
try:
    cfg_eval = BaselineConfig(
        data_root=DATA_ROOT,
        model_path=merged,
        out_dir=RUN_DIR,
        run_name=RUN_TAG,
        max_pixels=1280 * 720,
        seed=42,
        n_eval=40 if SMOKE else None,
    )
    assert cfg_eval.answer_postprocess is None, (
        "answer_postprocess must stay None -- rung 15's parser is a SECOND variable, "
        "and this run is the control")
    report = run_baseline(cfg_eval)
    print(f"eval done in {(time.perf_counter()-t0)/60:.1f} min")
finally:
    if MERGED_HERE and not SMOKE and Path(merged).is_dir():
        shutil.rmtree(merged, ignore_errors=True)
        print(f"reclaimed ~17 GB -> removed {merged}")

## Score canonically + GATE 0 on gold coverage

The floor understates (and the margin therefore overstates) when gold is incomplete —
`template_floor` drops ungolded rows from the numerator but keeps them in the denominator,
warning only via `logger.warning`. Coverage is asserted, not trusted.

In [ ]:
gold = ledger.gold_from_frame_parquets(DATA_ROOT)
res = pd.read_csv(RUN_DIR / RUN_TAG / "results.csv")
missing = set(res["qID"]) - set(gold.dropna(subset=["answer"])["qID"])
assert not missing, f"GATE 0 -- {len(missing)} qIDs without gold; margins would be inflated"
print(f"GATE 0: gold {len(res)}/{len(res)} OK")

metrics.assert_no_dup_qid(res); metrics.assert_ood_from_qid(res); metrics.assert_all_rows_grouped(res)
strat = metrics.stratified_report(res, gold=gold)
metrics.assert_floors_vs_eval_set(strat)

if not SMOKE:
    assert len(res) == 6252, f"expected the full eval set, got {len(res)} rows"
    ledger.register_run(RUN_DIR / RUN_TAG, strat,
                        experiment="06-vit-lora",
                        run=f"{RUN_DIR.name}__{RUN_TAG}",
                        model=f"b1_vit_lora epoch 3 ({CKPT})", date="2026-07-27")
print("gates OK   bucket_mean:", strat["bucket_mean"])

## The pre-registered read

Primary = `bucket_mean`. Secondary, always reported = `number` margin over the template-aware
floor, ID and OOD. **Never raw `acc_number`** (data card §3).

The verdict is read against the branches written down **before** the number existed:

- **ep3 ≥ 0.5699** → rung 15's ep3 win is not a win; both team rungs are closed, and rung 06
  may have gained a free checkpoint.
- **0.5667 ≤ ep3 < 0.5699** → rung 15's margin shrinks to whatever is left, still under the
  ID-AND-OOD conjunction it already fails.
- **ep3 < 0.5667** → epoch 3 costs rung 06 something, and the ep3 rows of rungs 14 and 15 need
  rereading against a control that is itself declining.

A human writes the verdict into the decision note. The table below only reports.

In [ ]:
REF = {
    "06 ep1 (checkpoint-860)":   0.5345110412460505,
    "06 ep2 (checkpoint-1720)":  0.566707285167315,
    "14 ep3 (appearance-aug)":   0.5610573044633793,
    "15 ep3 (count-target)":     0.5699318113206112,
}
bf = pd.DataFrame(strat["by_format"])
num = bf[bf.answer_format == "number"].set_index("distribution")
row = {
    "checkpoint": CKPT,
    "bucket_mean": strat["bucket_mean"],
    "acc_ID": strat["acc_ID"], "acc_OOD": strat["acc_OOD"],
    "margin_ID": strat["margin_ID"], "margin_OOD": strat["margin_OOD"],
    "number_margin_ID": num.loc["ID", "margin"] if "ID" in num.index else float("nan"),
    "number_margin_OOD": num.loc["OOD", "margin"] if "OOD" in num.index else float("nan"),
}
for name, v in REF.items():
    row[f"d_vs_{name.split()[0]}_{name.split()[1]}"] = strat["bucket_mean"] - v

out = pd.DataFrame([row])
if not SMOKE:
    out.to_csv(Path.cwd() / "RESULTS_epoch3.csv", index=False)
print(out.T.to_string())
print()
for name, v in REF.items():
    d = strat["bucket_mean"] - v
    print(f"  vs {name:28s} {v:.4f}   delta {d:+.4f}")